# 03 - RAG Simples
> Pipeline basico: indexar -> buscar -> responder

**Modulo:** `06_rag` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

model=SentenceTransformer('all-MiniLM-L6-v2')
col=chromadb.Client().get_or_create_collection('empresa')

DOCS=[
    'Ferias: 30 dias/ano, solicitar com 30 dias de antecedencia via RH.',
    'Home office: ate 3 dias/semana com aprovacao do gestor.',
    'Plano de saude: cobre consultas, exames e hospitalizacao. Dependentes com taxa.',
    'Reembolso: ate 30 dias apos o gasto, com nota fiscal no sistema Financas.',
]

embs=model.encode(DOCS).tolist()
col.upsert(ids=[f'p{i}'for i in range(len(DOCS))],embeddings=embs,documents=DOCS)

def rag(q,n=2):
    res=col.query(query_embeddings=model.encode([q]).tolist(),n_results=n)
    ctx='\n'.join(f'[Doc {i+1}]: {d}'for i,d in enumerate(res['documents'][0]))
    return ask(f'Use APENAS os docs abaixo.\n{ctx}\n\nPergunta: {q}',model=HAIKU)

for p in ['Quantos dias de ferias tenho?','O plano cobre minha familia?','Qual o salario inicial?']:
    print(f'Pergunta: {p}'); print(f'Resposta: {rag(p)}\n')

## Exercicios
> Complete os exercicios abaixo.


In [ ]:
# Seu codigo aqui


## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
